# LoRA GRPO Fine-Tuning on Ray with Training Hub on OpenShift AI

This notebook runs multi-node distributed **GRPO** training using Training Hub's verl backend on a Ray cluster. We train [Qwen/Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B) with LoRA to produce correct tool calls, using the [Agent-Ark/Toucan-1.5M](https://huggingface.co/datasets/Agent-Ark/Toucan-1.5M) dataset.

The verl backend distributes training across nodes using FSDP for the policy network and vLLM for inference. CodeFlare SDK submits the job as a `RayJob` with a `ManagedClusterConfig` — the SDK creates a short-lived RayCluster, runs the job, and tears everything down on completion.

For more details on GRPO and hardware requirements, see the [README](./README.md).

## Setup

Import the required dependencies.

In [ ]:
from codeflare_sdk import ManagedClusterConfig, RayJob
from kubernetes.client import (
    V1PersistentVolumeClaimVolumeSource,
    V1Volume,
    V1VolumeMount,
)

## Authenticate to your OpenShift Cluster

Log in via `oc login`. Get your token from `oc whoami -t` or the OpenShift console (Copy login command).

The `--insecure-skip-tls-verify` flag is needed for clusters with self-signed certificates.

> **Note:** The CodeFlare SDK's `RayJob` path reads authentication from the local kubeconfig file.
> Using `oc login` ensures the kubeconfig is written correctly for the SDK to pick up.

In [ ]:
!oc login --token=<YOUR_TOKEN> --server=<YOUR_API_SERVER> --insecure-skip-tls-verify

## 1. Configure Training Parameters

Key parameters:

### GRPO Parameters
- **num_iterations**: Number of GRPO iterations (each = rollout + train phase)
- **tasks_per_iteration**: Prompts sampled per iteration for rollout
- **group_size**: Responses generated per prompt for comparison
- **n_train**: Total training examples from dataset

### LoRA Parameters
- **lora_r**: Rank of the low-rank matrices (higher = more capacity, more memory)
- **lora_alpha**: Scaling factor

### vLLM Parameters
- **gpu_memory_utilization**: Fraction of GPU memory reserved for vLLM inference (remainder used by FSDP training)

### Cluster Configuration
- **NUM_WORKERS**: Number of Ray worker nodes (all GPUs are on worker nodes)
- **N_GPUS_PER_NODE**: GPUs per worker node
- **IMAGE**: Ray runtime image with Training Hub, verl, and vLLM pre-installed

In [ ]:
# Cluster configuration
NAMESPACE = "<YOUR_NAMESPACE>"  # Replace with your namespace
IMAGE = "quay.io/modh/ray@sha256:f63a015302758f805e5332605669b886e4d7ac60ec929413a2ffc19a904211c6"  # quay.io/modh/ray:2.55.1-py312-cu129-th081
NUM_WORKERS = 1  # Number of Ray worker nodes
N_GPUS_PER_NODE = 4  # GPUs per worker node (head has no GPUs)
HF_TOKEN = ""  # Optional: set your Hugging Face token (huggingface.co/settings/tokens)

# Storage — PVC shared across all pods (head + workers)
PVC_NAME = "shared"  # Replace if the shared RWX storage name is different than in the example provided
PVC_PATH = "shared"  # Replace if the shared RWX storage path is different than in the example provided
PVC_MOUNT_PATH = f"/opt/app-root/src/{PVC_PATH}"

# Model and dataset
MODEL_PATH = "Qwen/Qwen3-4B"
DATA_PATH = "Agent-Ark/Toucan-1.5M"
OUTPUT_DIR = f"{PVC_MOUNT_PATH}/grpo-output"

# GRPO hyperparameters
NUM_ITERATIONS = 5
TASKS_PER_ITERATION = 10
GROUP_SIZE = 4
N_TRAIN = 200
N_VAL = 10
LEARNING_RATE = 1e-5

# LoRA configuration
LORA_R = 16
LORA_ALPHA = 8

# vLLM configuration
GPU_MEMORY_UTILIZATION = 0.20

## 2. Configure the Ray Cluster

Configure the multi-node Ray cluster for distributed GRPO training. verl distributes
training across nodes using FSDP for the policy network and vLLM for inference.

The head node handles Ray coordination only (no GPUs). All training GPUs are
on the worker node, giving verl optimal single-node GPU topology for FSDP and vLLM.

A PVC is mounted on **all pods** so that training artifacts (reward function,
checkpoints, data) are accessible from every node.

In [ ]:
pvc_volume = V1Volume(
    name="training-data",
    persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(claim_name=PVC_NAME),
)
pvc_mount = V1VolumeMount(name="training-data", mount_path=PVC_MOUNT_PATH)

env_vars = {}
if HF_TOKEN:
    env_vars["HF_TOKEN"] = HF_TOKEN
    env_vars["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

cluster_config = ManagedClusterConfig(
    image=IMAGE,
    head_cpu_requests=4,
    head_cpu_limits=8,
    head_memory_requests=32,
    head_memory_limits=32,
    num_workers=NUM_WORKERS,
    worker_cpu_requests=8,
    worker_cpu_limits=16,
    worker_memory_requests=192,
    worker_memory_limits=256,
    worker_accelerators={"nvidia.com/gpu": N_GPUS_PER_NODE},
    volumes=[pvc_volume],
    volume_mounts=[pvc_mount],
    envs=env_vars,
)

## 3. Build the Training Entrypoint

The entrypoint calls `training_hub.lora_grpo()` with the verl backend.
Training Hub handles all verl orchestration internally — setting up Ray actors,
configuring FSDP, launching vLLM, and running the GRPO training loop.

In [ ]:
import base64

TOTAL_GPUS = N_GPUS_PER_NODE * NUM_WORKERS  # GPUs on worker nodes only

_code = f"""\
from training_hub import lora_grpo

result = lora_grpo(
    model_path='{MODEL_PATH}',
    data_path='{DATA_PATH}',
    ckpt_output_dir='{OUTPUT_DIR}',
    backend='verl',
    n_gpus={TOTAL_GPUS},
    num_iterations={NUM_ITERATIONS},
    tasks_per_iteration={TASKS_PER_ITERATION},
    group_size={GROUP_SIZE},
    n_train={N_TRAIN},
    n_val={N_VAL},
    lora_r={LORA_R},
    lora_alpha={LORA_ALPHA},
    learning_rate={LEARNING_RATE},
    gpu_memory_utilization={GPU_MEMORY_UTILIZATION},
)

print('Training complete')
print('Status:', result.get('status'))
print('Reward history:', result.get('reward_history'))
"""

_encoded = base64.b64encode(_code.encode()).decode()
entrypoint = (
    f"python -c \"import base64; exec(base64.b64decode('{_encoded}').decode())\""
)

print("Entrypoint configured")
print(f"Model: {MODEL_PATH}")
print(f"Dataset: {DATA_PATH}")
print(f"Nodes: 1 head + {NUM_WORKERS} worker(s)")
print(f"GPUs: {TOTAL_GPUS} total ({N_GPUS_PER_NODE} per worker, head has none)")
print(f"Iterations: {NUM_ITERATIONS}")

## 4. Submit the RayJob

The SDK creates a `RayJob` CR with an embedded `RayCluster` spec. KubeRay:
1. Spins up the Ray cluster (head for coordination + worker pods with GPUs)
2. Runs the entrypoint on the head node
3. Tears everything down when the job finishes (`shutdownAfterJobFinishes: true`)

The `ttl_seconds_after_finished` controls how long the RayJob CR persists after
completion (for log inspection). Set to 0 for immediate cleanup.

In [ ]:
job = RayJob(
    job_name="grpo-verl-training",
    entrypoint=entrypoint,
    cluster_config=cluster_config,
    namespace=NAMESPACE,
    ttl_seconds_after_finished=600,
)

job.submit()
print(f"RayJob '{job.name}' submitted to namespace '{NAMESPACE}'")

## 5. Monitor Training Progress

Poll the job status. The status may show as `PENDING` while the RayCluster is
spinning up and pulling the image across nodes.

GRPO training with the default parameters on 4× A100-80GB (2 nodes) takes approximately 15-30 minutes.

In [ ]:
job.status()

In [ ]:
import time

print("Waiting for job to complete...")
print(
    f"Expected duration: ~15-30 minutes for {NUM_ITERATIONS} iterations on {TOTAL_GPUS} GPUs\n"
)

for i in range(120):
    status, ready = job.status(print_to_console=False)
    print(f"[{i * 30}s] Status: {status.value}")
    if ready or status.value in ("FAILED", "COMPLETE"):
        break
    time.sleep(30)

print(f"\nFinal status: {status.value}")

## 6. View Job Logs

Retrieve the RayJob logs to inspect training output, reward scores, and any errors.

In [ ]:
job.logs()

## 7. Plot Reward Curve

Parse the reward history from the job logs and plot it. The mean reward should
increase over iterations as the model learns to produce correct tool calls.

In [ ]:
import re

import matplotlib.pyplot as plt

log_output = job.logs()

rewards = re.findall(r"Reward history:\s*\[([^\]]+)\]", log_output)

if rewards:
    reward_values = [float(r) for r in rewards[-1].split(",")]
    iterations = list(range(1, len(reward_values) + 1))

    plt.figure(figsize=(8, 4))
    plt.plot(iterations, reward_values, marker="o")
    plt.xlabel("Iteration")
    plt.ylabel("Mean Reward")
    plt.title("GRPO Training — Reward Curve")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Final reward: {reward_values[-1]:.4f}")
else:
    print("Could not parse reward history from logs.")
    print("Check job.logs() output above for training results.")

## 8. Cleanup

The RayCluster is automatically torn down when the job finishes
(`shutdownAfterJobFinishes: true`). The RayJob CR is cleaned up after
`ttl_seconds_after_finished` (600 seconds).

Use `job.delete()` for immediate cleanup if needed.

In [ ]:
# Uncomment to delete the RayJob immediately:
# job.delete()